In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from air_quality_monitor.analysis import AirQualityAnalyser
from air_quality_monitor.storage import CSVStorage
from air_quality_monitor.models import AirQualityReading
from pathlib import Path
import pandas as pd
from datetime import datetime
#import csv

In [ ]:
# Read in the data and add hour (0-23) & day of week columns (0-6) for aggregation
notebook_dir = Path().resolve()
filepath = notebook_dir.parent / "data" / "aqi_history.csv"

storage = CSVStorage(filepath, AirQualityReading)

df = storage.read()
df['hour'] = df['pollutant_timestamp'].dt.hour
df['dow'] = df['pollutant_timestamp'].dt.day_of_week
#print(df)
print(df.dtypes)

In [ ]:
# Plot line charts showing AQI over time for different cities
# First, get data for required time period for each city, and create DF
analyser = AirQualityAnalyser(df)
df_to_print = pd.DataFrame()
for city in ["Berlin", "Athens", "Stockholm"]:
    df_city = analyser.filter_by_city(city).get_aqi_by_date_range(start_date=datetime(2026, 3, 24, 9, 0))
    df_city['city'] = city
    print(df_city.shape)
    print(df_city.columns)
    df_to_print = pd.concat([df_to_print, df_city])

print(df_to_print)

In [ ]:
# Secondly, create and print the figure
fig = analyser.plot_line_chart(df_to_print, x_axis="pollutant_timestamp", y_axis="aqi", title="AQI over time")
fig.show()

In [ ]:
# Show the mean AQI per hour over the duration
# Focus on Sarajevo for now, and only the values we're interested in
df_sj = df[df['city'] == 'Sarajevo']
df_sj_aqihr = df_sj[['city', 'hour', 'aqi']]

# Aggregate by hour and get the mean AQI
df_sj_grouped = df_sj_aqihr.groupby(['city', 'hour']).mean().reset_index()
df_sj_grouped.rename(columns={'aqi': 'mean_aqi'}, inplace=True)

# And plot it
fig = analyser.plot_line_chart(df_sj_grouped, x_axis='hour', y_axis='mean_aqi', title="Mean AQI over time")
fig.show()

In [ ]:
# Now let's just look at weekdays (0 <= day <= 4)
df_sj_wd = df_sj[df_sj['dow'].between(0,4)]
df_sj_wd_aqihr = df_sj_wd[['city', 'hour', 'aqi']]
df_sj_wd_aqihr.info()

# And this is weekends (5 <= day <= 6)
df_sj_we = df_sj[df_sj['dow'].between(5,6)]
df_sj_we_aqihr = df_sj_we[['city', 'hour', 'aqi']]
df_sj_we_aqihr.info()



In [ ]:
# Then group by hour and get the mean AQI, then plot.

df_sj_wd_grouped = df_sj_wd_aqihr.groupby(['city', 'hour']).mean().reset_index()
df_sj_we_grouped = df_sj_we_aqihr.groupby(['city', 'hour']).mean().reset_index()

fig = analyser.plot_line_chart(df_sj_wd_grouped, x_axis='hour', y_axis='aqi', title="Mean AQI over time: Weekdays")
fig.show()

fig = analyser.plot_line_chart(df_sj_we_grouped, x_axis='hour', y_axis='aqi', title="Mean AQI over time: Weekends")
fig.show()

In [ ]:
# Let's add in London et al as well

df_aqihr_print = pd.DataFrame()
df_aqihr_wd_print = pd.DataFrame()
df_aqihr_we_print = pd.DataFrame()

for city in ["Sarajevo", "London", "Stockholm", "Athens"]:
    df_city = df[df['city'] == city]
    
    df_city_aqihr = df_city[['city', 'hour', 'aqi']]
    df_city_aqihr_grouped = df_city_aqihr.groupby(['city', 'hour']).mean().reset_index()
    df_aqihr_print = pd.concat([df_aqihr_print, df_city_aqihr_grouped])
    
    df_city_wd = df_city[df_city['dow'].between(0,4)]
    df_city_aqihr_wd = df_city_wd[['city', 'hour', 'aqi']]
    df_city_aqihr_wd_grouped = df_city_aqihr_wd.groupby(['city', 'hour']).mean().reset_index()
    df_aqihr_wd_print = pd.concat([df_aqihr_wd_print, df_city_aqihr_grouped])
    
    df_city_we = df_city[df_city['dow'].between(5,6)]
    df_city_aqihr_we = df_city_we[['city', 'hour', 'aqi']]
    df_city_aqihr_we_grouped = df_city_aqihr_we.groupby(['city', 'hour']).mean().reset_index()
    df_aqihr_we_print = pd.concat([df_aqihr_we_print, df_city_aqihr_grouped])
    
df_aqihr_print.rename(columns={'aqi': 'mean_aqi'}, inplace=True)
df_aqihr_wd_print.rename(columns={'aqi': 'mean_aqi'}, inplace=True)
df_aqihr_we_print.rename(columns={'aqi': 'mean_aqi'}, inplace=True)

# And plot it
for d in [df_aqihr_print, df_aqihr_wd_print, df_aqihr_we_print]:
    fig = analyser.plot_line_chart(d, x_axis='hour', y_axis='mean_aqi', title="Mean AQI over time")
    fig.show()